# Fabric: OneLake lakehouse → Direct Lake semantic model → parity check

Run this in a **Fabric notebook** attached to a lakehouse named `copilot_lakehouse` (left pane → Lakehouses → Add).

1. **Load** the certified gold tables into the lakehouse as Delta tables in OneLake.
2. **Create a Direct Lake semantic model as code** (semantic-link-labs), with the same metric definitions as the Databricks metric views, written in DAX.
3. **Parity check:** the DAX measures must return exactly what SQL returns on the same tables. A metric that means one thing in Genie and another in Power BI is the failure this prevents.
4. **Export the model as TMDL**, the text format Fabric Git integration stores.

**Production path (not used here):** a Mirrored Azure Databricks Catalog would expose the gold tables in OneLake without copying. That needs Azure Databricks; this project runs on Databricks Free Edition, so the gold tables are loaded from the project's GitHub snapshot (`data/gold/*.parquet`, synthetic data).

In [ ]:
# Parameters
GITHUB_RAW = "https://raw.githubusercontent.com/jjaishwin/agency-performance-copilot/main/data/gold"
TABLES = ["account_market_month", "agency_month", "agency_scorecard", "account_scorecard", "contract_scenarios",
          "account_briefs", "dim_account", "dim_market", "aadv_business_compliance", "target_points"]
LAKEHOUSE = "copilot_lakehouse"
MODEL = "Agency Performance (Direct Lake)"
MODEL_TABLES = ["account_market_month", "agency_month", "account_scorecard", "agency_scorecard", "target_points"]

## 1 · Load the gold tables into OneLake

In [ ]:
import pandas as pd
from pyspark.sql import functions as F

for t in TABLES:
    pdf = pd.read_parquet(f"{GITHUB_RAW}/{t}.parquet")
    df = spark.createDataFrame(pdf)
    for c in ("month", "quarter_start"):
        if c in df.columns:
            df = df.withColumn(c, F.to_date(c))
    df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(t)
    print(f"{t:<26} {df.count():>7,} rows")

In [ ]:
# Quick check: risk counts match the Databricks scorecard
display(spark.sql("SELECT risk_status, COUNT(*) AS accounts FROM account_scorecard GROUP BY risk_status ORDER BY accounts DESC"))

## 2 · Create the Direct Lake semantic model, as code

**Direct Lake** reads the Delta files in OneLake directly: no import copy to refresh, no DirectQuery round trip per visual. `use_sql_endpoint=False` asks for **Direct Lake on OneLake**; if the workspace doesn't support it, the cell falls back to Direct Lake on the SQL endpoint and says so.

Direct Lake models can't hold calculated tables, so the deal what-if uses the `target_points` table (0, 5, 10, 15, 20) as a disconnected slicer.

In [ ]:
%pip install semantic-link-labs --quiet

In [ ]:
import sempy_labs as labs

DIRECT_LAKE_MODE = "Direct Lake on OneLake"
try:
    labs.generate_direct_lake_semantic_model(dataset=MODEL, tables=MODEL_TABLES, source=LAKEHOUSE,
                                             source_type="Lakehouse", use_sql_endpoint=False,
                                             overwrite=True, refresh=False)
except Exception as e:
    print("Direct Lake on OneLake not available here:", str(e)[:300])
    DIRECT_LAKE_MODE = "Direct Lake on SQL endpoint"
    labs.generate_direct_lake_semantic_model(dataset=MODEL, tables=MODEL_TABLES, source=LAKEHOUSE,
                                             source_type="Lakehouse", use_sql_endpoint=True,
                                             overwrite=True, refresh=False)
print("Created", MODEL, "·", DIRECT_LAKE_MODE)

### Measures: the metric views, in DAX

Each measure matches a Databricks metric view measure (`notebooks/06_metric_views.py`) and `microsoft/powerbi/measures.dax`. Ratios divide sums; percentages are never averaged.

In [ ]:
AMM, AG = "account_market_month", "agency_month"
ELIGIBLE_PAX = "CALCULATE ( SUM ( account_market_month[aa_pax] ), account_market_month[qsi_eligible] = TRUE () )"

MEASURES = [
    # (table, name, DAX, format, description)
    (AMM, "AA Revenue", "SUM ( account_market_month[aa_revenue] )", "$#,0", "AA revenue in USD"),
    (AMM, "Passenger Share", "DIVIDE ( SUM ( account_market_month[aa_pax] ), SUM ( account_market_month[total_pax] ) )",
     "0.0%", "AA passengers divided by all-carrier passengers"),
    (AMM, "Fair Share (QSI)",
     "DIVIDE ( SUMX ( account_market_month, account_market_month[qsi_share] * account_market_month[total_pax] ), SUM ( account_market_month[total_pax] ) )",
     "0.0%", "Passenger-weighted QSI fair share. Never average QSI percentages."),
    (AMM, "Share Gap (pts)", "( [Passenger Share] - [Fair Share (QSI)] ) * 100", "+0.0;-0.0;0.0",
     "Passenger share minus fair share, in percentage points"),
    (AMM, "Contract Attainment",
     "DIVIDE ( " + ELIGIBLE_PAX + ", CALCULATE ( SUMX ( account_market_month, account_market_month[target_share] * account_market_month[total_pax] ), account_market_month[qsi_eligible] = TRUE () ) )",
     "0.0%", "Actual AA passengers over contracted target passengers, QSI-eligible markets. Below 90% = at risk."),
    (AMM, "At-Risk Accounts",
     "COUNTROWS ( FILTER ( VALUES ( account_market_month[account_id] ), NOT ISBLANK ( [Contract Attainment] ) && [Contract Attainment] < 0.90 ) )",
     "#,0", "Managed accounts below 90% contract attainment"),
    (AMM, "Premium Mix", "DIVIDE ( SUM ( account_market_month[aa_premium_revenue] ), SUM ( account_market_month[aa_revenue] ) )",
     "0.0%", "Share of AA revenue from premium cabins"),
    (AMM, "Target Points Value", "SELECTEDVALUE ( target_points[target_points], 10 )", "0",
     "Contract target being tested: fair share + this many points"),
    (AMM, "Scenario Target Pax",
     "SUMX ( FILTER ( account_market_month, account_market_month[qsi_eligible] ), MIN ( account_market_month[qsi_share] + [Target Points Value] / 100, 0.95 ) * account_market_month[total_pax] )",
     "#,0", "Passengers the tested target would require"),
    (AMM, "Scenario Attainment", "DIVIDE ( " + ELIGIBLE_PAX + ", [Scenario Target Pax] )", "0.0%",
     "Attainment if the contract were fair share + the selected points"),
    (AMM, "Scenario Passenger Gap", "MAX ( 0, [Scenario Target Pax] - " + ELIGIBLE_PAX + " )", "#,0",
     "Passengers short of the tested target"),
    (AG, "Agency Revenue Share", "DIVIDE ( SUM ( agency_month[aa_revenue] ), SUM ( agency_month[total_revenue] ) )",
     "0.0%", "AA revenue share at the agency"),
    (AG, "NDC Share", "DIVIDE ( SUM ( agency_month[aa_ndc_bookings] ), SUM ( agency_month[aa_bookings] ) )",
     "0.0%", "NDC bookings over all AA bookings"),
    (AG, "Revenue per Incentive $", "DIVIDE ( SUM ( agency_month[aa_revenue] ), SUM ( agency_month[incentive_cost_usd] ) )",
     "$#,0.00", "AA revenue per incentive dollar"),
]

In [ ]:
from sempy_labs.tom import connect_semantic_model

with connect_semantic_model(dataset=MODEL, readonly=False) as tom:
    for table, name, expr, fmt, desc in MEASURES:
        tom.add_measure(table_name=table, measure_name=name, expression=expr, format_string=fmt, description=desc)
    # Slice account-level measures by scorecard attributes (risk status, region of the account)
    tom.add_relationship(from_table="account_market_month", from_column="account_id",
                         to_table="account_scorecard", to_column="account_id",
                         from_cardinality="Many", to_cardinality="One", cross_filtering_behavior="OneDirection")
print(f"Added {len(MEASURES)} measures and 1 relationship")

labs.refresh_semantic_model(dataset=MODEL)  # for Direct Lake this is "framing": point the model at the latest Delta version

## 3 · Parity check: DAX must equal SQL on the same tables

The same question is answered twice: by the DAX measures through the semantic model, and by SQL on the lakehouse. Any difference fails the cell.

In [ ]:
import sempy.fabric as fabric

def compare(name, dax, sql, key, cols, tol=1e-9):
    pbi = fabric.evaluate_dax(MODEL, dax)
    pbi.columns = [key] + cols
    ref = spark.sql(sql).toPandas()
    m = pbi.merge(ref, on=key, suffixes=("_dax", "_sql"))
    worst = max((m[f"{c}_dax"].astype(float) - m[f"{c}_sql"].astype(float)).abs().max() for c in cols)
    ok = len(m) == len(ref) == len(pbi) and worst <= tol
    print(f"{'PASS' if ok else 'FAIL'}  {name}: {len(m)} rows, largest difference {worst:.2e}")
    return ok, m

checks = []
ok, by_region = compare(
    "share vs fair share by region, 2026",
    """EVALUATE SUMMARIZECOLUMNS ( account_market_month[region],
         FILTER ( ALL ( account_market_month[month] ), account_market_month[month] >= DATE ( 2026, 1, 1 ) ),
         "passenger_share", [Passenger Share], "fair_share", [Fair Share (QSI)], "gap_pts", [Share Gap (pts)] )""",
    """SELECT region, SUM(aa_pax) / SUM(total_pax) AS passenger_share,
              SUM(qsi_share * total_pax) / SUM(total_pax) AS fair_share,
              (SUM(aa_pax) - SUM(qsi_share * total_pax)) / SUM(total_pax) * 100 AS gap_pts
       FROM account_market_month WHERE month >= '2026-01-01' GROUP BY region""",
    "region", ["passenger_share", "fair_share", "gap_pts"])
checks.append(ok)

ok, by_am = compare(
    "contract attainment by account manager, managed, 2026",
    """EVALUATE SUMMARIZECOLUMNS ( account_market_month[account_manager],
         FILTER ( ALL ( account_market_month[month] ), account_market_month[month] >= DATE ( 2026, 1, 1 ) ),
         TREATAS ( { "Managed" }, account_market_month[segment] ),
         "attainment", [Contract Attainment] )""",
    """SELECT account_manager,
              SUM(CASE WHEN qsi_eligible THEN aa_pax END)
                / SUM(CASE WHEN qsi_eligible THEN target_share * total_pax END) AS attainment
       FROM account_market_month WHERE month >= '2026-01-01' AND segment = 'Managed' GROUP BY account_manager""",
    "account_manager", ["attainment"])
checks.append(ok)

display(by_region.sort_values("gap_pts_dax"))
display(by_am.sort_values("attainment_dax").head(5))
assert all(checks), "DAX and SQL disagree: fix the measure before anyone builds a report on it"
print("Power BI and SQL agree on every number checked.")

## 4 · Export the model as TMDL

TMDL is the text format of a semantic model: one file per table, measures as readable blocks. Fabric Git integration stores models this way, so every measure change is a reviewable diff. The files are written to the lakehouse under `Files/tmdl/`.

In [ ]:
import os

parts = labs.get_semantic_model_definition(MODEL, format="TMDL", return_dataframe=False)
out_dir = "/lakehouse/default/Files/tmdl"
for p in parts:
    path = os.path.join(out_dir, p["file_name"])
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(p["content"])
print(f"Wrote {len(parts)} TMDL files to Files/tmdl/")

amm = next(p["content"] for p in parts if p["file_name"].endswith("tables/account_market_month.tmdl"))
start = amm.find("measure 'Contract Attainment'")
print(amm[start:start + 700] if start >= 0 else amm[:700])

## Next: the report (about 15 minutes, in the browser)

Open the semantic model `Agency Performance (Direct Lake)` → **Create report** (blank). See `microsoft/FABRIC_POWERBI_STEPS.md` for the exact visuals.